In [1]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model = 'gemini-2.5-flash',
    contents = 'Hey, how are U?'
)

print(response.text)

I'm doing well, thank you for asking! As an AI, I don't experience emotions, but I'm fully operational and ready to assist.

How are you doing today?


In [2]:
def llm(prompt):
    response = client.models.generate_content(
        model = 'gemini-2.5-flash',
        contents = prompt
    )
    return response.text

In [9]:
llm("Do you know about the round of 32 teams for the world cup?")

That's an interesting question, and it depends on whether you're talking about past/current World Cup formats or the upcoming 2026 World Cup!

1.  **For past and current World Cups (e.g., 2022 and prior):**
    *   The tournament **starts with 32 teams** divided into 8 groups of 4.
    *   After the group stage, the top two teams from each group (8 groups x 2 teams = **16 teams total**) advance to the knockout stage.
    *   The **first knockout round is called the "Round of 16."**
    *   So, there isn't a "Round of 32" as a *knockout stage* in the traditional 32-team World Cup format. The 32 teams are the ones *in the group stage*.

2.  **For the 2026 World Cup (USA, Canada, Mexico):**
    *   This will be the first World Cup with **48 teams** participating!
    *   These 48 teams will be divided into 12 groups of 4.
    *   The top two teams from each group, *plus the 8 best third-placed teams*, will advance. This will result in **32 teams** moving on to the knockout stage.
    *   

In [5]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

That's great news! We're excited you discovered the course.

To give you the most accurate answer, I'll need a little more information about **which specific course** you're referring to.

Many courses, especially self-paced online ones, allow continuous enrollment so you can join anytime. However, others might be:

*   **Cohort-based** with specific start dates (and you might need to wait for the next session).
*   **Live classes** that have already begun (and joining late might not be ideal or possible).
*   Courses with **application deadlines** or specific enrollment periods.

**Could you please tell me:**

1.  **What is the exact name of the course?**
2.  **Where did you discover it or where is it hosted?** (e.g., Coursera, Udemy, edX, a specific university website, a company's learning platform, etc.)

Once I have that, I can check for you and provide precise instructions on how to enroll or what the next steps are!


That was a very generic answer, so we need to add more context so the LLM can give a better response

In [3]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [6]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [7]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted.


## FAQ Dataset

In [8]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [9]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [10]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Let's fit the index for text search. 

- **Text fields** are the fields we search through. When we type a query, the search engine looks at these fields and tokenizes them. It splits text into words, lowercases them, removes stop words, and ranks the results by relevance.

- We use **Keyword fields** to restrict the search space to a particular subset. It's like the WHERE clause in a SQL query.

In [12]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': "llm-zoomcamp"},
    num_results= 5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

Wrapping the search in a function

In [18]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course":course}

    return index.search(
        question,
        boost_dict= boost_dict,
        filter_dict= filter_dict,
        num_results= 5
    )

In [22]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Let's build a prompt

We usually split the prompt in two parts: 

- Instructions (system prompt) to tell the LLM how to behave. It never changes, so it's the same for every request.

- User prompts. This changes with every request. It carries the actual question and the retrieved context.


In [23]:
INSTRUCTIONS = """
    Your task is to answer questions from the course participants based on the provided context.capitalize

    Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."
"""

In [31]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [25]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [28]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [32]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### Querying the LLM with the prompt we just created

In [33]:
response = client.models.generate_content(
    model = 'gemini-2.5-flash',
    contents = prompt
)

In [ ]:
response.candidates[0].content.parts[0]

Part(
  text="""Yes, you can join now.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who finish the course with a "live" cohort, as it requires peer-reviewing projects while the course is running.

You can start learning and submitting homework (while the form is open) without formal registration, as the course materials are available and you can begin whenever you like."""
)

In [53]:
print(response.text)

Yes, you can join now.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who finish the course with a "live" cohort, as it requires peer-reviewing projects while the course is running.

You can start learning and submitting homework (while the form is open) without formal registration, as the course materials are available and you can begin whenever you like.


In [55]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Yes, you can join now.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who finish the course with a "live" cohort, as it requires peer-reviewing projects while the course is running.

You can start learning and submitting homework (while the form is open) without formal registration, as the course materials are available and you can begin whenever you like."""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='T_VBatGFJLGM_PUP5pbPkAE',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_coun

LLM function

In [58]:
from google.genai import types

def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    response = client.models.generate_content(
    model = model,
    config=types.GenerateContentConfig(
        system_instruction= instructions    
    ),
    contents = user_prompt
)
    
    return response.text

In [60]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    
    return answer

In [61]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.


In [66]:
answer = rag("How many live classes are there in the course")
print(answer)

The context states that there are no separate live sessions for every module by default. Live sessions are announced separately when they happen. The provided text does not specify the exact number of live classes or sessions in the course.
